# QuranMediaLib — Getting Started

An **interactive** walkthrough. Run each code cell top-to-bottom. Most cells have an **`# ── EDIT ME ──`** block at the top — change a surah number, an ayah, a translation string, or an effect parameter, then re-run the cell to **see what happens**.

Each render is a real `PIL.Image`. The image from one step flows into the next, so you build up a finished artwork cell-by-cell.

## Step 0 — Setup

Run this once. It opens the bundled Quran databases (text, word-by-word, translations) and loads the layout presets.

In [ ]:
from IPython.display import Markdown, display

from quranmedialib import (
    LANDSCAPE_PRESET,
    SQUARE_PRESET,
    STORY_PRESET,
    DatabaseManager,
    SurahWorkflow,
    VerseWorkflow,
)
from quranmedialib.modules.image import color, glow, pad
from quranmedialib.modules.wimage import get_wimage
from quranmedialib.types import Padding

# open the bundled databases once for this kernel
db = DatabaseManager()

PRESETS = {"landscape": LANDSCAPE_PRESET, "story": STORY_PRESET, "square": SQUARE_PRESET}

print("Setup ready — databases open, presets loaded.")

## Step 1 — Render a single verse

**Edit the numbers/string, then re-run this cell.** The rendered verse appears inline.

- `SURAH` — chapter number (1–114). 108 = Al-Kawthar, 1 = Al-Fatiha, 112 = Al-Ikhlas.
- `AYAH` — verse number within that surah.
- `LAYOUT` — `"landscape"` (16:9), `"story"` (9:16), or `"square"` (1:1).
- `MODE` — `"default"` (annotated + translation), `"arabic"` (Arabic only), or `"translation"` (translation only).
- `TRANSLATIONS` — one string per page. Keep it short for a 1-page verse; edit it to describe the verse you picked.

In [ ]:
# ── EDIT ME: change these, then re-run ─────────────────────────────
SURAH = 108  # chapter number (1-114)
AYAH = 1  # verse number within the surah
LAYOUT = "landscape"  # "landscape" | "story" | "square"
MODE = "default"  # "default" | "arabic" | "translation"
TRANSLATIONS = ["Indeed, We have granted you, [O Muhammad], al-Kawthar."]
RESOLUTION = "1080p"

preset = PRESETS[LAYOUT][MODE][RESOLUTION]
workflow = VerseWorkflow(preset)
pages = list(
    workflow.get_iterator(
        surah=SURAH,
        ayah=AYAH,
        translations=TRANSLATIONS,
        annotate=(MODE == "default"),
    )
)

# get_iterator yields one list of images per page; flatten them
renderings = [img for page in pages for img in page]
img = renderings[0]

print(f"Surah {SURAH} · ayah {AYAH} · {LAYOUT}/{MODE} · {len(renderings)} image(s)")
display(img)

## Step 2 — Post-process the render (the image flows through)

The `img` from **Step 1** continues into this cell. We pass it through the `glow` effect, add transparent padding, and `color` a single Arabic word mask to show the colorize behavior. **Edit `GLOW_*` / `PAD`, then re-run.**

In [ ]:
# ── EDIT ME: restyle the SAME verse ────────────────────────────────
GLOW_RGB = (0, 120, 255)  # halo color, 0-255 per channel
GLOW_STRENGTH = 1.3  # >1 more vibrant, <1 faded
GLOW_RADIUS = 70  # halo spread in pixels
PAD = 40  # transparent padding added around the frame

# this cell consumes `img` produced by the Step 1 cell
styled = glow(img, strength=GLOW_STRENGTH, radius=GLOW_RADIUS)
styled = pad(styled, Padding(PAD, PAD, PAD, PAD))
print(f"styled: {img.size[0]}x{img.size[1]} -> {styled.size[0]}x{styled.size[1]}")
display(Markdown("**Styled verse** (glow + padding):"))
display(styled)

# colorize a single word: get_wimage returns an 'L' luminance mask,
# and `color` fills that mask with the requested color
word_mask = get_wimage("\u0627\u0644\u0644\u0651\u0647", preset.word)  # Allah
word_color = color(word_mask, (255, 215, 0, 255))  # gold
display(Markdown("**Colorized word mask** (Arabic text -> gold fill):"))
display(word_color)

## Step 3 — Slice the words of a verse

Fetch the verse's Arabic words from the database, then **index-slice them** with Python slicing. Change `START` / `STOP` / `STEP` and re-run to see exactly which words render and how the line changes. The selected words are composed into a single right-to-left line (first word on the right).

In [ ]:
from PIL import Image

# ── EDIT ME: slice the same verse's words ──────────────────────────
START = 0  # first word index (0-based)
STOP = 4  # exclusive end index -> words[START:STOP]
STEP = 1  # take every STEP-th word

verse_text = db.get_verse(SURAH, AYAH)
words = verse_text.split()
selected = words[START:STOP:STEP]

print(f"Surah {SURAH} ayah {AYAH} has {len(words)} words:")
for i, word in enumerate(words):
    marker = "  <-- selected" if i in range(START, STOP, STEP) else ""
    print(f"  [{i:2d}] {word}{marker}")

if not selected:
    raise ValueError("The slice selected zero words — check START / STOP / STEP.")


# compose the selected words into one RTL line (first word on the right)
def compose_words(word_texts, word_config, spacing=13) -> Image.Image:
    images = [get_wimage(w, word_config) for w in word_texts]
    total_w = sum(im.width for im in images) + spacing * (len(images) - 1)
    max_h = max(im.height for im in images)
    canvas = Image.new("RGBA", (total_w, max_h), (0, 0, 0, 0))
    x = total_w  # start at the right edge (RTL)
    for im in images:
        x -= im.width
        canvas.paste(im, (x, (max_h - im.height) // 2), mask=im)
        x -= spacing
    return canvas


line = compose_words(selected, preset.word)
display(Markdown(f"**Selected {len(selected)} word(s)** as one RTL line:"))
display(line)

## Step 4 — Scale up to a whole surah

`SurahWorkflow` fetches every verse and a translation automatically. **Edit `SURAH_FULL`, then re-run.** Al-Ikhlas (112) renders 4 pages; Al-Fatiha (1) renders several.

In [ ]:
# ── EDIT ME ─────────────────────────────────────────────────────────
SURAH_FULL = 112  # try 1 (Al-Fatiha) or 112 (Al-Ikhlas)

surah_preset = PRESETS["landscape"]["default"]["1080p"]
surah_workflow = SurahWorkflow(surah_preset)
surah_pages = list(surah_workflow.get_iterator(surah=SURAH_FULL, annotate=True))
surah_images = [img for page in surah_pages for img in page]

print(f"Surah {SURAH_FULL} -> {len(surah_images)} page image(s)")
for i, page in enumerate(surah_images, 1):
    display(Markdown(f"**page {i}**"))
    display(page)

## Step 5 — Save your artwork

Saves the styled verse (from Step 2) to disk. Re-run **Step 1 → Step 2 → Step 5** after editing to export a new design.

In [ ]:
from pathlib import Path

OUT_DIR = Path("output/notebook")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# save the styled version if Step 2 ran, otherwise the plain render
artwork = styled if "styled" in globals() else img
save_path = OUT_DIR / "my_verse.png"
artwork.save(save_path)
print(f"Saved {save_path.resolve()}")

## Clean up

Close the databases. Run this when you are done.

In [ ]:
db.close()
print("Databases closed.")